# Likelihood-family and prediction verification

This notebook inspects BayesBreak's concrete likelihood families on deterministic synthetic signals, compares MAP segmentations, verifies posterior predictive densities against analytic references, and records support violations as expected outcomes.

Outputs are written to `results/notebook_verification/families/`. Family comparisons illustrate software behavior on controlled fixtures; they are not evidence that one family is universally preferable.

In [ ]:
from __future__ import annotations

import inspect
import json
import platform
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_distribution
from scipy.stats import betabinom

import bayesbreak
from bayesbreak import (
    BayesBreakBernoulli,
    BayesBreakBeta,
    BayesBreakBinomial,
    BayesBreakGaussian,
    BayesBreakPoisson,
    make_bayesbreak,
)
from bayesbreak.prediction import posterior_predictive_logpdf

SEED = 20260810
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
OUTPUT_DIR = ROOT / "results" / "notebook_verification" / "families"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

print({"python": platform.python_version(), "bayesbreak": bayesbreak.__version__, "seed": SEED})
print(f"Report directory: {OUTPUT_DIR.relative_to(ROOT)}")

## 1. Inspect the family factory and prediction interface

The factory provides aliases for concrete estimators. Posterior prediction routes new observations through each fitted family's `posterior_predictive_logpdf_block` implementation, so unsupported families must fail explicitly rather than falling back to a Gaussian density.

In [ ]:
family_names = ["gaussian", "poisson", "bernoulli", "binomial", "beta", "negbin"]
interface_frame = pd.DataFrame(
    [
        {
            "family": name,
            "class": type(make_bayesbreak(name)).__name__,
            "constructor": str(inspect.signature(type(make_bayesbreak(name)))),
            "predictive_implemented": type(make_bayesbreak(name)).posterior_predictive_logpdf_block.__qualname__.split(".")[0] != "BayesBreakSegmenter",
        }
        for name in family_names
    ]
)
display(interface_frame)
assert interface_frame.predictive_implemented.all()

## 2. Create family-specific fixtures and fit the models

All fixtures contain one change at index 40 but obey different supports. Binomial trial counts are explicit descriptors, and Beta observations remain strictly inside $(0,1)$.

In [ ]:
rng = np.random.default_rng(SEED)
n = 80
coordinates = np.arange(n, dtype=float).reshape(-1, 1)
true_boundary = 40
trial_counts = np.full(n, 10.0)

fixtures = {
    "gaussian": np.r_[rng.normal(-1.0, 0.35, 40), rng.normal(1.5, 0.35, 40)],
    "poisson": np.r_[rng.poisson(2.0, 40), rng.poisson(10.0, 40)].astype(float),
    "bernoulli": np.r_[rng.binomial(1, 0.15, 40), rng.binomial(1, 0.85, 40)].astype(float),
    "binomial": np.r_[rng.binomial(10, 0.2, 40), rng.binomial(10, 0.75, 40)].astype(float),
    "beta": np.r_[rng.beta(3.0, 12.0, 40), rng.beta(12.0, 3.0, 40)],
}
models = {
    "gaussian": BayesBreakGaussian(k_max=6),
    "poisson": BayesBreakPoisson(k_max=6),
    "bernoulli": BayesBreakBernoulli(k_max=6),
    "binomial": BayesBreakBinomial(k_max=6, n_trials=trial_counts),
    "beta": BayesBreakBeta(k_max=6, concentration=20.0),
}

fit_rows = []
for family, model in models.items():
    started = time.perf_counter()
    model.fit(coordinates, fixtures[family])
    fit_rows.append(
        {
            "family": family,
            "k_map": model.k_map_,
            "boundaries": model.map_boundaries_,
            "closest_boundary_error": min(abs(boundary - true_boundary) for boundary in model.map_boundaries_[1:-1]),
            "log_evidence": model.log_evidence_,
            "posterior_entropy": float(-np.sum(model.k_posterior_[model.k_posterior_ > 0] * np.log(model.k_posterior_[model.k_posterior_ > 0]))),
            "fit_ms": 1000 * (time.perf_counter() - started),
        }
    )
fit_frame = pd.DataFrame(fit_rows)
display(fit_frame)
assert all(np.isclose(model.k_posterior_.sum(), 1.0) for model in models.values())
assert (fit_frame.closest_boundary_error <= 5).all()

## 3. Visualize MAP signals and posterior segment counts

Rows align the observed support, fitted MAP signal, declared boundary, and posterior over segment count. The axes intentionally differ by family because their response scales are not commensurate.

In [ ]:
fig, axes = plt.subplots(len(models), 2, figsize=(12, 13))
colors = {"gaussian": "#00798C", "poisson": "#D1495B", "bernoulli": "#30638E", "binomial": "#EDAE49", "beta": "#2A9D8F"}
for row, (family, model) in enumerate(models.items()):
    ax = axes[row, 0]
    ax.scatter(coordinates[:, 0], fixtures[family], s=12, alpha=0.55, color="#555555", label="observed")
    ax.plot(coordinates[:, 0], model.map_curve_, linewidth=2, color=colors[family], label="MAP signal")
    ax.axvline(true_boundary, color="#D1495B", linestyle="--", linewidth=1, label="declared boundary")
    ax.set(title=family.capitalize(), xlabel="index", ylabel="response")
    if row == 0:
        ax.legend(ncol=3, fontsize=8)
    ax = axes[row, 1]
    k_values = np.arange(1, model.k_posterior_.size + 1)
    ax.bar(k_values, model.k_posterior_, color=colors[family])
    ax.axvline(model.k_map_, color="#333333", linestyle="--", linewidth=1)
    ax.set(title=f"{family}: P(k | y)", xlabel="segments k", ylabel="probability")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "family_fits.png", bbox_inches="tight")
plt.show()

## 4. Verify posterior prediction against analytic references

These checks isolate one fitted segment so the conjugate posterior is available in closed form. Bernoulli uses Beta–Bernoulli prediction, Binomial uses the Beta-Binomial PMF with explicit new-trial descriptors, and fractional Beta uses the declared Beta density.

In [ ]:
prediction_rows = []

bernoulli_train = np.array([1.0, 0.0, 1.0])
bernoulli = BayesBreakBernoulli(k_max=1, estimate_hyper=False, alpha=2.0, beta=3.0).fit(np.arange(3), bernoulli_train)
bernoulli_new = np.array([0.0, 1.0])
bernoulli_observed = posterior_predictive_logpdf(bernoulli, np.array([0.0, 2.0]), bernoulli_new, per_sample=True)
posterior_probability = (2.0 + bernoulli_train.sum()) / (5.0 + bernoulli_train.size)
bernoulli_expected = np.log([1.0 - posterior_probability, posterior_probability])
prediction_rows.append({"family": "bernoulli", "max_abs_error": float(np.max(np.abs(bernoulli_observed - bernoulli_expected))), "observed": bernoulli_observed.tolist(), "expected": bernoulli_expected.tolist()})

binomial_train = np.array([2.0, 4.0])
binomial = BayesBreakBinomial(k_max=1, estimate_hyper=False, n_trials=np.array([5.0, 5.0]), alpha=2.0, beta=3.0).fit(np.arange(2), binomial_train)
new_successes = np.array([1.0, 3.0])
new_trials = np.array([2.0, 4.0])
binomial_observed = posterior_predictive_logpdf(binomial, np.array([0.0, 1.0]), new_successes, sample_weight=new_trials, per_sample=True)
binomial_expected = betabinom.logpmf(new_successes, new_trials, 8.0, 7.0)
prediction_rows.append({"family": "binomial", "max_abs_error": float(np.max(np.abs(binomial_observed - binomial_expected))), "observed": binomial_observed.tolist(), "expected": binomial_expected.tolist()})

beta_train = np.array([0.2, 0.4])
beta_model = BayesBreakBeta(k_max=1, estimate_hyper=False, concentration=10.0, alpha=2.0, beta=3.0).fit(np.arange(2), beta_train)
beta_new = np.array([0.3, 0.7])
beta_observed = posterior_predictive_logpdf(beta_model, np.array([0.0, 1.0]), beta_new, per_sample=True)
beta_expected = beta_distribution.logpdf(beta_new, 8.0, 17.0)
prediction_rows.append({"family": "beta", "max_abs_error": float(np.max(np.abs(beta_observed - beta_expected))), "observed": beta_observed.tolist(), "expected": beta_expected.tolist()})

prediction_frame = pd.DataFrame(prediction_rows)
display(prediction_frame)
assert (prediction_frame.max_abs_error < 1e-10).all()

## 5. Plot posterior predictive distributions

The plots show actual predictive mass or density, obtained by exponentiating the verified log scores. Binomial predictions condition on the displayed number of new trials.

In [ ]:
beta_grid = np.linspace(0.01, 0.99, 200)
beta_density = np.exp(posterior_predictive_logpdf(beta_model, np.zeros(beta_grid.size), beta_grid, per_sample=True))
binomial_grid = np.arange(0, 6, dtype=float)
binomial_mass = np.exp(posterior_predictive_logpdf(binomial, np.zeros(6), binomial_grid, sample_weight=np.full(6, 5.0), per_sample=True))
bernoulli_mass = np.exp(bernoulli_observed)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].bar([0, 1], bernoulli_mass, color="#30638E")
axes[0].set(title="Beta–Bernoulli predictive", xlabel="new outcome", ylabel="probability", xticks=[0, 1])
axes[1].bar(binomial_grid, binomial_mass, color="#EDAE49")
axes[1].set(title="Beta-Binomial predictive (5 trials)", xlabel="new successes", ylabel="probability")
axes[2].plot(beta_grid, beta_density, color="#2A9D8F", linewidth=2)
axes[2].fill_between(beta_grid, beta_density, color="#2A9D8F", alpha=0.2)
axes[2].set(title="Fractional-Beta predictive", xlabel="new value", ylabel="density")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "predictive_distributions.png", bbox_inches="tight")
plt.show()
assert np.isclose(bernoulli_mass.sum(), 1.0)
assert np.isclose(binomial_mass.sum(), 1.0)

## 6. Capture support and unsupported-operation failures

Support checks are part of scientific correctness: binary data cannot silently become fractional, Binomial trial descriptors must be positive integers, Beta observations cannot be clipped onto the boundary, and logistic-normal posterior prediction is explicitly unimplemented.

In [ ]:
from bayesbreak import BayesBreakLogisticNormal

logistic = BayesBreakLogisticNormal(k_max=2, approx="quadrature").fit(
    np.arange(8), np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0])
)
expected_failures = [
    ("bernoulli nonbinary", lambda: posterior_predictive_logpdf(bernoulli, np.array([1.0]), np.array([0.5])), ValueError, "{0, 1}"),
    ("binomial fractional trials", lambda: posterior_predictive_logpdf(binomial, np.array([0.0]), np.array([1.0]), sample_weight=np.array([2.5])), ValueError, "positive integers"),
    ("beta boundary zero", lambda: posterior_predictive_logpdf(beta_model, np.array([0.0]), np.array([0.0])), ValueError, "strictly in"),
    ("logistic prediction unsupported", lambda: posterior_predictive_logpdf(logistic, np.array([2.0]), np.array([1.0])), NotImplementedError, "does not implement"),
]
error_rows = []
for name, operation, expected_type, message_fragment in expected_failures:
    started = time.perf_counter()
    try:
        operation()
        error_rows.append({"case": name, "status": "fail", "exception": "none", "message": "", "elapsed_ms": 1000 * (time.perf_counter() - started), "traceback": ""})
    except Exception as error:
        matched = isinstance(error, expected_type) and message_fragment in str(error)
        error_rows.append({"case": name, "status": "pass" if matched else "fail", "exception": type(error).__name__, "message": str(error), "elapsed_ms": 1000 * (time.perf_counter() - started), "traceback": traceback.format_exc() if not matched else ""})
error_frame = pd.DataFrame(error_rows)
display(error_frame[["case", "status", "exception", "message", "elapsed_ms"]])
assert (error_frame.status == "pass").all()

## 7. Compare runtime and generate the report

Runtime is shown only for these matched-size fixtures and this machine. The final report combines fitted-model checks, analytic prediction error, and expected support failures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
axes[0].bar(fit_frame.family, fit_frame.fit_ms, color=["#00798C", "#D1495B", "#30638E", "#EDAE49", "#2A9D8F"])
axes[0].set(title="Fit runtime by family", xlabel="family", ylabel="milliseconds")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(prediction_frame.family, prediction_frame.max_abs_error, color="#2A9D8F")
axes[1].axhline(1e-10, color="#D1495B", linestyle="--", label="tolerance")
axes[1].set(title="Analytic predictive agreement", xlabel="family", ylabel="maximum absolute log-score error", yscale="symlog", ylim=(-1e-16, 1e-8))
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "verification_summary.png", bbox_inches="tight")
plt.show()

fit_frame.to_csv(OUTPUT_DIR / "family_fits.csv", index=False)
prediction_frame.to_json(OUTPUT_DIR / "analytic_prediction_checks.json", orient="records", indent=2)
error_frame.to_csv(OUTPUT_DIR / "expected_failures.csv", index=False)
report = {
    "bayesbreak_version": bayesbreak.__version__,
    "seed": SEED,
    "families_fitted": list(models),
    "all_posteriors_normalized": bool(all(np.isclose(model.k_posterior_.sum(), 1.0) for model in models.values())),
    "analytic_predictions_passed": bool((prediction_frame.max_abs_error < 1e-10).all()),
    "support_contracts_passed": bool((error_frame.status == "pass").all()),
}
report["all_required_checks_passed"] = all([report["all_posteriors_normalized"], report["analytic_predictions_passed"], report["support_contracts_passed"]])
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["all_required_checks_passed"]